<a id="seccion0"></a>
### Segmentación de imágenes SEM del meteorito

### **Tabla de Contenidos**
- [ 1 - Importación de librerías relevantes y funciones](#seccion1)
  - [ 1.1 - Librerías](#seccion11)
  - [ 1.2 - Funciones](#seccion12)

- [ 2 - Seleccion de la región de interes y guardado](#seccion2)
    - [ 2.1 - Seleccion ROI](#seccion21)
    - [ 2.2 - Guardado ROI](#seccion22)

- [ 3 - Segmentado](#seccion3)
    - [ 3.1 - Cargar Datos guardados ROI](#seccion31)
    - [ 3.2 - Procesamiento](#seccion32)
      - [ 3.2.1 - Blureo](#seccion321)
      - [ 3.2.2 - Corrección de Sombras Locales](#seccion322)
      - [ 3.2.3 - Filtro No Lineal - Tangente Hiperbólica](#seccion323)
      - [ 3.2.4 - Binarizado Adaptativo](#seccion324)
      - [ 3.2.5 - Eliminación de Motas](#seccion325)

- [ 4 - Analisis de la segmentación](#seccion4)

<a id="seccion1"></a>
### **1 - Importación  de librerías relevantes y funciones**

<a id="seccion11"></a>
#### **1.1 - Librerias**
Volver a la [Tabla de contenidos.](#seccion0)

In [ ]:
# /usr/bin/env python sirve para indicar que el script debe ejecutarse con el intérprete de Python. Esto permite que el script se ejecute en cualquier sistema donde Python esté instalado, sin necesidad de especificar la ruta completa del intérprete.

In [ ]:
#! /usr/bin/env python
import numpy as np
import matplotlib.pyplot as plt
import cv2 # psci
from pathlib import Path  # pathlib es independiente del sistema operativo
from matplotlib.widgets import RectangleSelector

<a id="seccion12"></a>
### 1.2 - Funciones

Volver a la [tabla de contenidos.](#seccion0)

In [ ]:
def seleccionar_roi(nombre_archivo, ruta_imagenes, parametros_roi):
    """
    Permite al usuario seleccionar un rectángulo (ROI) en la imagen especificada usando una ventana interactiva de matplotlib.
    
    """
    ruta_archivo = Path(ruta_imagenes) / Path(nombre_archivo)
    imagen = np.asarray(cv2.imread(ruta_archivo, cv2.IMREAD_GRAYSCALE))
    fig, ax = plt.subplots()
    ax.imshow(imagen, cmap="gray" if imagen.ndim == 2 else None)
    ax.set_title(f"Seleccionar el rectángulo en {nombre_archivo}")

    def al_seleccionar(inicio, fin):
        """
        Función llamada cuando se selecciona un rectángulo para guardar las coordenadas del ROI.
        """
        if inicio.xdata is None or fin.xdata is None:
            return

        x1, x2 = sorted((round(inicio.xdata), round(fin.xdata)))
        y1, y2 = sorted((round(inicio.ydata), round(fin.ydata)))

        parametros_roi[nombre_archivo] = (x1, y1, x2, y2)

    selector = RectangleSelector(
        ax,
        al_seleccionar,
        useblit=True,
        button=[1],
        interactive=True,
    )
    plt.show()
    return imagen, selector

<a id="seccion2"></a>
### **2 - Seleccion de la región de interes y guardado**

* ```%matplotlib widget``` permite interactuar con la imagen y seleccionar la región de interes (ROI). El paquete para esta funcion es **ipympl**

<a id="section21"></a>
#### **2.1 - Seleccion ROI**
Volver a la [ Tabla de contenidos.](#seccion0)

In [ ]:
# ventana interactiva para seleccionar el ROI
%matplotlib widget 

#diccionario con cada archivo y sus coordenadas de ROI (x1, y1, x2, y2)
ruta_imagenes = "imagenes_SEM"
parametros_roi = {
    "20260520_2_07.tif": (0, 0, 0, 0),
    "20260520_2_08.tif": (0, 0, 0, 0),
    "20251219_Meteorito1_04.tif": (0, 0, 0, 0),
    "20251229_metorito_01.tif": (0, 0, 0, 0),
    "20251229_metorito_03.tif": (0, 0, 0, 0),
}

# Seleccionar ROI para un archivo específico
nombre_archivo = "20251219_Meteorito1_04.tif"
imagen, selector = seleccionar_roi(nombre_archivo, ruta_imagenes, parametros_roi)


<a id="seccion22"></a>
#### **2.2 - Guardado ROI**

##### - Se gurada un archivo .npy y un png
##### - Cuando el codigo esté terminado, recordar cambiar ```sufijo``` de la ROI cada vez que se genere una zona nueva
Volver a la [ Tabla de contenidos.](#seccion0)

In [ ]:
# -----------------------------------------------------------------
# cambiar el sufijo del archivo para imagenes nuevas
# -----------------------------------------------------------------

sufijo = "_1"  # sufijo para el nombre del archivo de salida

# preparacion del recorte

x1, y1, x2, y2 = parametros_roi[nombre_archivo]
recorte = imagen[y1:y2, x1:x2]
#print(f"shape : {recorte.shape}, dtype : {recorte.dtype}")
#print(f"Recorte de la imagen {nombre_archivo} con ROI: {parametros_roi[nombre_archivo]}")

# guardado del recorte:

# .npy
nombre_base = nombre_archivo.stem  # .stem quita la extensión
ruta_mascaras = Path("mascaras_roi") / Path(nombre_base) / Path(nombre_base + sufijo)
ruta_mascaras.mkdir(parents=True, exist_ok=True)
ruta_archivo = ruta_mascaras / f"roi_{nombre_base}{sufijo}.npy"
np.save(ruta_archivo, recorte)

# .tif
ruta_imagen_tif = ruta_mascaras / f"roi_{nombre_base}{sufijo}.tif"
Image.fromarray(recorte).save(ruta_imagen_tif)


##### **Graficación de recortes fuera de la ventana interactiva**

In [ ]:
# salida de la ventana interactiva y mostrar el recorte de la imagen
%matplotlib inline

In [ ]:
# grafica del recorte de la imagen
#plt.imshow(recorte, cmap="gray" if recorte.ndim == 2 else None) 
#plt.show()

<a id = "seccion3"></a>
#### **3 - Segmentado**

Volver a la [Tabla de contenido](#seccion0)

<a id="seccion31"></a>
##### **3.1 - Cargar Datos guardados ROI .npy**

Aquí se cargan las ROI .npy guardadas previamente

Los archivos cargables son:

- "20260520_2_07.tif"
- "20260520_2_08.tif"
- "20251219_Meteorito1_04.tif"
- "20251229_metorito_01.tif"
- "20251229_metorito_03.tif"

In [ ]:
# carga de matriz ROI desde archivo .npy
nombre_archivo = "20251219_Meteorito1_04.tif"
nombre_base = Path(nombre_archivo).stem
sufijo = "_1"  # sufijo para el nombre del archivo de salida
ruta_mascaras = Path("mascaras_roi") / Path(nombre_base) / Path(nombre_base + sufijo)
ruta_archivo = ruta_mascaras / f"roi_{nombre_base}{sufijo}.npy"
roi = np.load(ruta_archivo)

# fig0, ax0 = plt.subplots()
# ax0.imshow(roi, cmap="gray" if roi.ndim == 2 else None)

<a id ="seccion32"></a>
#### **3.2 - Procesamiento**

[3.2.1 Blureo](#seccion321)

[3.2.2 Corrección de Sombras Locales](#seccion322)

[3.2.3 Filtro No Lineal - Tangente Hiperbólica](#seccion323)

[3.2.4 Binarizado Adaptativo](#seccion324)

[3.2.5 Eliminación de Motas](#seccion325)

In [ ]:
roi.shape

In [ ]:
# Operaciones realizadas sobre la matriz ROI
blureo_local = True
correccion_sombras_locales = True
filtro_no_lineal = True
binarizado = True
correccion_motas = True

<a id="seccion321"></a>
##### **3.2.1 Blureo**

In [ ]:
if blureo_local:
    """
    Esta sección del código aplica un desenfoque (blur) a la imagen recortada (ROI) utilizando un kernel de tamaño (5, 5). El desenfoque suaviza la imagen, reduciendo el ruido y las variaciones de intensidad, lo que puede ser útil para mejorar la calidad de la imagen ante las operaciones de procesamiento.
    """
    kernel_blur = (3, 3)
    
    roi_blur = cv2.blur(roi, kernel_blur, 0).astype(np.float32)  # el tercer argumento es el ancla del kernel (yo le diría pivote). El 0 indica que se calculará automáticamente y lo coloca en el centro.

    fig1, ax1 = plt.subplots()
    ax1.imshow(roi, cmap="viridis" if roi.ndim == 2 else None)
else:
    roi_blur = roi.copy().astype(np.float32)

In [ ]:
roi.sa

<a id="seccion322"></a>
##### **3.2.2 Correcion de Sombras Locales**

Volver al indice de [Procesamiento](#seccion32)

In [ ]:
if correccion_sombras_locales:
    """
    Esta sección del código realiza la corrección de sombras locales en la imagen recortada (ROI) utilizando desenfoque y resta. La idea es suavizar la imagen para obtener una versión "promediada" de la misma, que representa las sombras y variaciones de iluminación, y luego restar esta versión suavizada de la imagen original para resaltar los detalles y eliminar las sombras.
    """
    fraccion = 0.03 # fraccion del lado de la imagen
    particion_imagen = int(fraccion*np.sqrt(roi.size))
    print(f"tamaño del kernel: ({particion_imagen}, {particion_imagen})")
    kernel_sombras = (particion_imagen, particion_imagen)
    
    sombras = cv2.blur(roi_blur, kernel_sombras)
    roi_sin_sombras = roi_blur - sombras

    fig2, ax2 = plt.subplots(1,2)
    ax2[0].imshow(roi_blur, cmap="gray" if roi_blur.ndim == 2 else None)
    ax2[0].set_title("Imagen antes")
    ax2[1].imshow(roi_sin_sombras, cmap="gray" if roi_sin_sombras.ndim == 2 else None)
    ax2[1].set_title("Imagen sin sombras")
else:
    roi_sin_sombras = roi_blur.copy()

<a id="seccion323"></a>
##### **3.2.3 Filtro No Lineal - Tangente Hiperbólica**

Volver al indice de [Procesamiento](#seccion32)

In [ ]:
if filtro_no_lineal:
    """
    Normalización con tanh para mejorar el contraste de la imagen. La función tanh mapea los valores de entrada a un rango entre -1 y 1. Al aplicar esta función a la imagen corregida, se mejora el contraste y se resaltan los detalles en las áreas de intensidad media, mientras que las áreas muy oscuras o muy brillantes se comprimen.
    """
    roi_tanh = np.tanh(roi_sin_sombras / np.max(np.abs(roi_sin_sombras)))

    fig3, ax3 = plt.subplots(1,2)
    ax3[0].imshow(roi_sin_sombras, cmap="gray" if roi_sin_sombras.ndim == 2 else None)
    ax3[0].set_title("Imagen sin sombras")
    ax3[1].imshow(roi_tanh, cmap="gray" if roi_tanh.ndim == 2 else None)
    ax3[1].set_title("Imagen con tanh")
else:
    roi_tanh = roi_sin_sombras.copy()

<a id="seccion324"></a>
##### **3.2.4 Binarizado Adaptativo**

Volver al indice de [Procesamiento](#seccion32)

In [ ]:
if binarizado:
    """
    Binarización usando un umbral adaptativo

    El umbral adaptativo es una técnica de procesamiento de imágenes que permite segmentar una imagen en regiones binarias (blanco y negro) basándose en la intensidad local de los píxeles. A diferencia del umbral global, que utiliza un único valor de umbral para toda la imagen, el umbral adaptativo calcula un umbral diferente para cada píxel o región de la imagen, teniendo en cuenta las variaciones locales de iluminación y contraste. La forma general de esta funcion es:
    bin = cv2.adaptativeThreshold(src, maxValue, adaptiveMethod, thresholdType, blockSize, C)
    Donde
    - src: imagen de entrada
    
    - maxValue: valor máximo que se asigna a los píxeles que superan el umbral

    - adaptiveMethod: método de adaptación (cv2.ADAPTIVE_THRESH_MEAN_C o cv2.ADAPTIVE_THRESH_GAUSSIAN_C)
    
    - thresholdType: tipo de umbral (cv2.THRESH_BINARY o cv2.THRESH_BINARY_INV)
    
    - blockSize: tamaño del bloque para calcular el umbral local, debe ser impar y mayor que 1.
    
    - C: valor constante que se resta del umbral calculado
    """
    roi_uint8 = ((roi_tanh + 1) * 127.5).astype(np.uint8) # desplazamos, reescalamos y convertimos a tipo uint8 para el input de adaptativeThreshold \\ 127.5 = 255 / 2
    
    # print(roi_tanh.min(), roi_tanh.max())
    print(roi_uint8.min(), roi_uint8.max(), roi_uint8.dtype)
    
    block_size = 11
    roi_binarizada = cv2.adaptiveThreshold(
        roi_uint8, 255, cv2.ADAPTIVE_THRESH_MEAN_C, cv2.THRESH_BINARY, block_size, 9
    ).astype(np.float32) # 255 representa la altura media de la tanh desplazada y reescalada.

    fig4, ax4 = plt.subplots(1,2)
    ax4[0].imshow(roi_tanh, cmap="gray" if roi_tanh.ndim == 2 else None)
    ax4[0].set_title("Imagen con tanh")
    ax4[1].imshow(roi_binarizada, cmap="gray" if roi_binarizada.ndim == 2 else None)
    ax4[1].set_title("Imagen binarizada")
else:
    roi_binarizada = roi_tanh.copy()

<a id="seccion325"></a>
##### **3.2.5 Eliminacion de Motas**

Volver al indice de [Procesamiento](#seccion32)

In [ ]:
if correccion_motas:
    """
    Esta sección del código realiza la corrección de motas en la imagen binarizada utilizando la función connectedComponentsWithStats de OpenCV. La idea es identificar las regiones conectadas en la imagen binarizada y eliminar aquellas que tengan un área menor a un umbral definido (area_minima). Esto ayuda a limpiar la imagen de pequeñas motas o ruido que puedan estar presentes.
    """

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
    roi_binarizada, connectivity=8) # num_labels: número de etiquetas encontradas, labels: matriz de etiquetas, stats: estadísticas de cada componente, _ : centroides (no se utiliza en este caso).

    area_minima = 500  # píxeles; ajústalo según el tamaño de las motas
    roi_sin_motas = np.zeros_like(roi_binarizada) # imagen vacia para guardar la imagen sin motas.

    for etiqueta in range(1, num_labels):  # 0 es el fondo
        area = stats[etiqueta, cv2.CC_STAT_AREA]
        if area >= area_minima:
            roi_sin_motas[labels == etiqueta] = 255

<a id="seccion4"></a>
#### **4 - Analisis de la segmentación**
Volver a la [Tabla de contenidos](#seccion0)